# Codex Python SDK Walkthrough (Advanced)

This notebook focuses on practical, production-oriented usage patterns.

## 1) Sync flow with lifecycle-safe reads and retries

In [ ]:
from codex_app_server import Codex, TextInput, ImageInput
from codex_app_server.retry import retry_on_overload

with Codex() as codex:
    thread = codex.thread_start(model='gpt-5')
    _ = codex.thread_read(thread.id, include_turns=False)

    result = retry_on_overload(
        lambda: thread.turn(TextInput('Give 3 bullets about resilient SDK design.')).run(),
        max_attempts=3,
    )
    print('status=', result.status)

    vision = thread.turn([
        TextInput('Describe this image in 2 bullets.'),
        ImageInput('https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg'),
    ]).run()
    print('vision status=', vision.status)

## 2) Async parity with explicit turn completion guards

In [ ]:
import asyncio
from codex_app_server.async_client import AsyncAppServerClient

async def run_async_demo():
    async with AsyncAppServerClient() as client:
        await client.initialize()
        started = await client.thread_start(model='gpt-5')
        tid = started['thread']['id']

        turn = await client.turn_text(tid, 'One sentence on backpressure.')
        turn_id = turn['turn']['id']

        chunks = []
        while True:
            evt = await client.next_notification()
            if evt.method == 'item/agentMessage/delta':
                chunks.append((evt.params or {}).get('delta', ''))
            if evt.method == 'turn/completed' and (evt.params or {}).get('turn', {}).get('id') == turn_id:
                break

        print(''.join(chunks).strip())

await run_async_demo()